### Data Source and Limitations\n\n**Note:** The paper uses a proprietary dataset from Finaeon, which includes 1, 2, 5, and 10-year government bond rates for all ten currencies. As this dataset is not publicly available, this notebook uses public data sources as a substitute:\n\n- **FX Rates:** Pulled from Yahoo Finance.\n- **Bond Yields:** Pulled from the FRED API.\n\nThis leads to a key limitation: the FRED database provides 1, 2, 5, and 10-year yields only for USD. For all other currencies, only the 10-year yield is consistently available. The code adapts to this by using the 10-year yield as a proxy where shorter-term yields are required for non-USD currencies.

# FX Rate Prediction with Graph Learning\n\nThis notebook implements the data fetching and preprocessing steps described in the paper \"Graph Learning for Foreign Exchange Rate Prediction and Statistical Arbitrage\" (arXiv:2508.14784). It prepares the data for a graph-based machine learning model to predict foreign exchange rates.\n\nThe notebook is designed to be run on Google Colab.

## 0. Setup

In [ ]:
!pip install -q yfinance pandas numpy requests scikit-learn torch torch_geometric

## 1. Data Fetching\n\nIn this section, we fetch the necessary data from our two sources:\n\n- **FRED (Federal Reserve Economic Data):** For government bond yields.\n- **Yahoo Finance:** For foreign exchange (FX) rates.

In [ ]:
import yfinance as yf\nimport pandas as pd\nimport numpy as np\nimport requests\nfrom datetime import datetime

### 1.1. FRED Data for Bond Yields

In [ ]:
FRED_API_KEY = '8f503a1e7348fa967987e5ad187992b9'\n\nBOND_SERIES = {\n    'USD': {'1Y': 'DGS1', '2Y': 'DGS2', '5Y': 'DGS5', '10Y': 'DGS10'},\n    'EUR': {'10Y': 'IRLTLT01EZM156N'},\n    'JPY': {'10Y': 'IRLTLT01JPM156N'},\n    'GBP': {'10Y': 'IRLTLT01GBM156N'},\n    'AUD': {'10Y': 'IRLTLT01AUM156N'},\n    'CAD': {'10Y': 'IRLTLT01CAM156N'},\n    'CHF': {'10Y': 'IRLTLT01CHM156N'},\n    'HKD': {'10Y': 'IRLTLT01HKM156N'},\n    'SGD': {'10Y': 'IRLTLT01SGM156N'},\n    'SEK': {'10Y': 'IRLTLT01SEM156N'}\n}

In [ ]:
def fetch_fred_data(series_id, api_key, start_date='1995-01-01', end_date='2024-12-31'):\n    url = f'https://api.stlouisfed.org/fred/series/observations?series_id={series_id}&api_key={api_key}&file_type=json&observation_start={start_date}&observation_end={end_date}'\n    response = requests.get(url)\n    data = response.json()\n    df = pd.DataFrame(data['observations'])\n    df = df[['date', 'value']]\n    df['date'] = pd.to_datetime(df['date'])\n    df = df.set_index('date')\n    df['value'] = pd.to_numeric(df['value'], errors='coerce')\n    return df

### 1.2. Yahoo Finance Data for FX Rates

In [ ]:
CURRENCIES = ['EUR', 'JPY', 'GBP', 'AUD', 'CAD', 'CHF', 'HKD', 'SGD', 'SEK']\nFX_TICKERS = [f'{currency}USD=X' for currency in CURRENCIES] + ['USDJPY=X']

In [ ]:
def fetch_fx_data(tickers, start_date='1995-01-01', end_date='2024-12-31'):\n    data = yf.download(tickers, start=start_date, end=end_date)['Close']\n    return data

### 1.3. Execute Data Fetching

In [ ]:
bond_data = {}\nfor currency, series in BOND_SERIES.items():\n    bond_data[currency] = {}\n    for term, series_id in series.items():\n        print(f'Fetching {currency} {term} bond yield...')\n        bond_data[currency][term] = fetch_fred_data(series_id, FRED_API_KEY)\n\nprint('\nFetching FX rates...')\nfx_data = fetch_fx_data(FX_TICKERS)\n\nprint('\nBond Data:')\nfor currency, data in bond_data.items():\n    for term, df in data.items():\n        print(f'{currency} {term}: {df.shape}')\n\nprint('\nFX Data:')\nprint(fx_data.shape)\nprint(fx_data.head())

## 2. Data Preprocessing and Feature Engineering\n\nIn this section, we preprocess the raw data to make it suitable for our graph model. The main steps are:\n\n1.  **Resampling:** The bond yield data for non-USD currencies is monthly. We need to upsample it to a daily frequency to match the FX data. We will use forward-filling for this.\n2.  **Merging:** Combine the bond yields and FX rates into a single DataFrame.\n3.  **Handling Missing Values:** Fill any remaining missing values.

In [ ]:
daily_index = pd.date_range(start=fx_data.index.min(), end=fx_data.index.max(), freq='D')\n\nprocessed_bond_data = {}\nfor currency, terms in bond_data.items():\n    for term, df in terms.items():\n        col_name = f'{currency}_{term}_yield'\n        # Reindex and forward-fill\n        processed_bond_data[col_name] = df.reindex(daily_index).ffill()\n\nbond_yields_df = pd.concat(processed_bond_data, axis=1)\n\nprint('Processed Bond Yields Data:')\nprint(bond_yields_df.shape)\nprint(bond_yields_df.head())

In [ ]:
df_full = pd.concat([fx_data, bond_yields_df], axis=1)\ndf_full = df_full.ffill()\n\nprint('Shape before cleaning:', df_full.shape)\n\n# Specific data cleaning as mentioned in the paper\n# Note: The dates and values might not exactly match due to using public data.\ndf_full.loc['2014-12-29':'2015-05-20', 'SGDAUD=X'] = np.nan # Repeated entries\ndf_full.loc[df_full['SEKEUR=X'] < 0.069, 'SEKEUR=X'] = np.nan # Erroneous value\ndf_full.loc[df_full.index.isin(['2023-04-24', '2024-01-12', '2024-01-26']) & (df_full['AUDCHF=X'] > 0.85), 'AUDCHF=X'] = np.nan # Erroneous values\ndf_full.loc['2022-01-01':'2023-12-31', ['HKD_10Y_yield', 'SGD_10Y_yield']] = df_full.loc['2022-01-01':'2023-12-31', ['HKD_10Y_yield', 'SGD_10Y_yield']].apply(lambda x: np.where(x > 90, np.nan, x))\n\ndf_full = df_full.ffill()\ndf_full = df_full.dropna()\n\nprint('Shape after cleaning:', df_full.shape)\nprint(df_full.head())

## 3. Time-Series Graph Preparation\n\nThis section implements the feature engineering process (`h_PI`) described in the paper to create a sequence of spatiotemporal graphs. Each graph in the sequence represents the state of the market at a specific time `t`, with features computed from look-back windows.

In [ ]:
from scipy.linalg import lstsq\nimport torch\nfrom torch_geometric.data import Data\n\ndef calculate_currency_values(data_t, currencies):\n    currency_map = {name: i for i, name in enumerate(currencies)}\n    num_currencies = len(currencies)\n    edges = []\n    for i, c1 in enumerate(currencies):\n        for j, c2 in enumerate(currencies):\n            if i < j:\n                # Need to construct the ticker and check if it exists\n                ticker1 = f'{c1}{c2}=X' if c2 != 'USD' else f'{c1}USD=X'\n                if c1 == 'USD': ticker1 = f'{c2}USD=X'\n                if ticker1 not in data_t.index and f'{c2}{c1}=X' in data_t.index:\n                    ticker1 = f'{c2}{c1}=X'\n                if ticker1 in data_t.index and not pd.isna(data_t[ticker1]):\n                    edges.append((c1, c2))\n\n    num_edges = len(edges)\n    A = np.zeros((num_edges + 1, num_currencies))\n    b = np.zeros(num_edges + 1)\n\n    for i, (c1, c2) in enumerate(edges):\n        A[i, currency_map[c1]] = 1\n        A[i, currency_map[c2]] = -1\n        ticker = f'{c1}{c2}=X' if c2 != 'USD' else f'{c1}USD=X'\n        if c1 == 'USD': ticker = f'{c2}USD=X'\n        if ticker not in data_t.index and f'{c2}{c1}=X' in data_t.index:\n            b[i] = -np.log(data_t[f'{c2}{c1}=X'])\n        else:\n            b[i] = np.log(data_t[ticker])\n\n    A[num_edges, :] = 1\n    b[num_edges] = 0\n\n    log_V, _, _, _ = lstsq(A, b)\n    return pd.Series(np.exp(log_V), index=currencies)\n\ndef get_all_fx_tickers(currencies):\n    tickers = []\n    for i in range(len(currencies)):\n        for j in range(i + 1, len(currencies)):\n            c1, c2 = currencies[i], currencies[j]\n            # yfinance uses specific conventions\n            if f'{c1}{c2}=X' in fx_data.columns:\n                tickers.append(f'{c1}{c2}=X')\n            elif f'{c2}{c1}=X' in fx_data.columns:\n                tickers.append(f'{c2}{c1}=X')\n    return list(set(tickers))\n\ndef generate_graph_sequence(df, currencies, lookback_windows):\n    graph_sequence = []\n    all_fx_tickers = get_all_fx_tickers(currencies)\n    df_v = pd.DataFrame({t: calculate_currency_values(df.loc[t], currencies) for t in df.index}).T\n    \n    for t in range(max(lookback_windows), len(df)):\n        current_time = df.index[t]\n        \n        # Node Features\n        node_features = []\n        for currency in currencies:\n            y_features = []\n            v_features = []\n            for window in lookback_windows:\n                # Interest Rate Features (y_ti) - using 1-year yield as per paper\n                # For non-USD, we use 10Y as a proxy since 1Y is not available.\n                ir_col = f'{currency}_1Y_yield' if currency == 'USD' else f'{currency}_10Y_yield'\n                if ir_col in df.columns:\n                    ir_log_diff = np.log(1 + df[ir_col].iloc[t-window:t]).diff().mean()\n                    y_features.append(ir_log_diff)\n                else:\n                    y_features.append(0) # Append 0 if data not available\n                \n                # Currency Value Features (v_ti)\n                v_log_diff = np.log(df_v[currency].iloc[t-window:t]).diff().mean()\n                v_features.append(v_log_diff)\n            node_features.append(np.concatenate([np.nan_to_num(y_features), np.nan_to_num(v_features)]).flatten())\n        node_features = torch.tensor(node_features, dtype=torch.float)\n        \n        # Edge Features (x_tij)\n        edge_features = []\n        num_nodes = len(currencies)\n        edge_index_list = []\n        for i in range(num_nodes):\n            for j in range(num_nodes):\n                if i == j: continue\n                edge_index_list.append([i, j])\n                x_feature = []\n                for window in lookback_windows:\n                    # Simplified: using USD based rates\n                    ticker = f'{currencies[i]}{currencies[j]}=X'\n                    if currencies[j] == 'USD': ticker = f'{currencies[i]}USD=X'\n                    if currencies[i] == 'USD': ticker = f'{currencies[j]}USD=X'\n                    if ticker in df.columns:\n                        fx_log_diff = np.log(df[ticker].iloc[t-window:t]).diff().mean()\n                        x_feature.append(fx_log_diff)\n                    else: # If direct ticker doesn't exist, append 0\n                        x_feature.append(0)\n                edge_features.append(np.nan_to_num(x_feature))\n        edge_features = torch.tensor(edge_features, dtype=torch.float)\n        edge_index = torch.tensor(edge_index_list, dtype=torch.long).t().contiguous()\n\n        # Target (y_t): log difference of next day's FX rate\n        # This part is for training, so we'll just create a placeholder\n        y = torch.zeros(edge_features.shape[0], 1, dtype=torch.float)\n\n        graph_sequence.append(Data(x=node_features, edge_index=edge_index, edge_attr=edge_features, y=y))\n    return graph_sequence\n\nlookback_windows = [1, 3, 5, 10, 15, 20]\ngraph_sequence = generate_graph_sequence(df_full, currencies, lookback_windows)\nprint(f"Generated {len(graph_sequence)} graphs for the time series.")\nif graph_sequence:\n    print("Example graph data object:")\n    print(graph_sequence[0])

## 4. Spatiotemporal GNN Model Implementation\n\nThis section implements the spatiotemporal Graph Neural Network (GNN) for FX rate prediction, as described in Section 4.1 of the paper. This model replaces the simple placeholder GNN.

In [ ]:
import torch.nn as nn\nimport torch.nn.functional as F\nfrom torch_geometric.nn import MessagePassing

In [ ]:
class SLP(nn.Module):\n    def __init__(self, in_features, out_features):\n        super(SLP, self).__init__()\n        self.linear = nn.Linear(in_features, out_features)\n        self.leaky_relu = nn.LeakyReLU()\n\n    def forward(self, x):\n        return self.leaky_relu(self.linear(x))

In [ ]:
class FXRP_GNN_Layer(MessagePassing):\n    def __init__(self, node_in_dim, edge_in_dim, node_out_dim, edge_out_dim):\n        super(FXRP_GNN_Layer, self).__init__(aggr='mean')\n        self.slp_node = SLP(node_in_dim + edge_in_dim + node_in_dim, node_out_dim)\n        self.slp_edge = SLP(node_out_dim + edge_in_dim + node_out_dim, edge_out_dim)\n\n    def forward(self, x, edge_index, edge_attr):\n        # Node update (propagate messages)\n        node_features_new = self.propagate(edge_index, x=x, edge_attr=edge_attr)\n        # Edge update\n        row, col = edge_index\n        edge_features_new = self.slp_edge(torch.cat([node_features_new[row], edge_attr, node_features_new[col]], dim=1))\n        return node_features_new, edge_features_new\n\n    def message(self, x_i, x_j, edge_attr):\n        # x_i: target node features, x_j: source node features\n        tmp = torch.cat([x_i, edge_attr, x_j], dim=1)\n        return self.slp_node(tmp)

In [ ]:
class FXRP_GNN(nn.Module):\n    def __init__(self, num_node_features, num_edge_features, hidden_dim, num_layers):\n        super(FXRP_GNN, self).__init__()\n        self.node_embedding = nn.Linear(num_node_features, hidden_dim)\n        self.edge_embedding = nn.Linear(num_edge_features, hidden_dim)\n\n        self.layers = nn.ModuleList()\n        for _ in range(num_layers):\n            self.layers.append(FXRP_GNN_Layer(hidden_dim, hidden_dim, hidden_dim, hidden_dim))\n\n        self.final_slp = nn.Linear(hidden_dim, 1) # No activation for the final layer\n\n    def forward(self, data):\n        node_features, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr\n\n        # Initial embedding\n        node_emb = self.node_embedding(node_features)\n        edge_emb = self.edge_embedding(edge_attr)\n\n        # GNN layers\n        for layer in self.layers:\n            node_emb, edge_emb = layer(node_emb, edge_index, edge_emb)\n\n        # Final prediction\n        return self.final_slp(edge_emb)\n\n# Instantiate the model (example hyperparameters)\nif graph_sequence:\n    model = FXRP_GNN(num_node_features=graph_sequence[0].num_node_features, \n                     num_edge_features=graph_sequence[0].num_edge_features, \n                     hidden_dim=64, \n                     num_layers=3)\n    print("Spatiotemporal GNN Model Architecture:")\n    print(model)

## 5. FXSA Graph and Feature Preparation\n\nThis section implements the feature engineering (`h_SI`) for the second GNN model (`f_S`), which is used for statistical arbitrage. It creates a new graph where nodes are the currency exchanges themselves.

In [ ]:
from scipy.linalg import null_space\nfrom itertools import permutations\n\ndef generate_fxsa_graph_sequence(predictions_sequence, all_currencies, lookback_windows, o='USD', epsilon_S=1e-4):\n    fxsa_graph_sequence = []\n    u_nodes = list(permutations(all_currencies, 2))\n    u_map = {name: i for i, name in enumerate(u_nodes)}\n    num_u_nodes = len(u_nodes)\n\n    alpha_hat_history = []\n    P_hat_history = []\n\n    for t in range(len(predictions_sequence)):\n        predictions_t = predictions_sequence.iloc[t]\n        V_hat_t = calculate_currency_values(predictions_t, all_currencies)\n        alpha_hat_t = {}\n        for i,j in u_nodes:\n            ticker = f'{i}{j}=X' if j != 'USD' else f'{i}USD=X'\n            if i == 'USD': ticker = f'{j}USD=X'\n            if ticker in predictions_t:\n                alpha_hat_t[(i,j)] = np.log(predictions_t[ticker]) - np.log(V_hat_t[i]) + np.log(V_hat_t[j])\n            else: alpha_hat_t[(i,j)] = 0\n        alpha_hat_history.append(alpha_hat_t)\n\n        constraint_matrix = []\n        for i in all_currencies:\n            if i == o: continue\n            row = np.zeros(num_u_nodes)\n            for j in all_currencies:\n                if i != j: row[u_map[(i,j)]] = 1\n            constraint_matrix.append(row)\n        for i_idx, i in enumerate(all_currencies):\n            for j_idx, j in enumerate(all_currencies):\n                if i_idx < j_idx:\n                    row = np.zeros(num_u_nodes)\n                    # Correctly implement Eq. 26: X_toi * u_tij + X_toj * X_tji * u_tji = 0\n                    X_hat_toi = predictions_t.get(f'{o}{i}=X'.replace('USDUSD','USD'), 1.0) if i != o else 1.0\n                    X_hat_toj = predictions_t.get(f'{o}{j}=X'.replace('USDUSD','USD'), 1.0) if j != o else 1.0\n                    \n                    # Handle inverse tickers for X_hat_ji\n                    ticker_ji = f'{j}{i}=X'.replace('USDUSD','USD')\n                    ticker_ij = f'{i}{j}=X'.replace('USDUSD','USD')\n                    if ticker_ji in predictions_t:\n                        X_hat_ji = predictions_t[ticker_ji]\n                    elif ticker_ij in predictions_t:\n                        X_hat_ji = 1.0 / predictions_t[ticker_ij]\n                    else:\n                        X_hat_ji = 1.0 # Fallback\n\n                    row[u_map[(i,j)]] = X_hat_toi\n                    row[u_map[(j,i)]] = X_hat_toj * X_hat_ji\n                    constraint_matrix.append(row)\n        B_hat_t = null_space(np.array(constraint_matrix))\n        P_hat_t = torch.tensor(B_hat_t @ np.linalg.pinv(B_hat_t.T @ B_hat_t) @ B_hat_t.T, dtype=torch.float)\n        P_hat_history.append(P_hat_t)\n\n    for t in range(max(lookback_windows), len(predictions_sequence)):\n        # Node features are temporal averages of alpha_hat\n        node_features = []\n        for i,j in u_nodes:\n            alpha_features = []\n            for window in lookback_windows:\n                avg_alpha = np.mean([alpha_hat_history[t-k][(i,j)] for k in range(window)])\n                alpha_features.append(avg_alpha)\n            node_features.append(alpha_features)\n        node_features = torch.tensor(node_features, dtype=torch.float)\n\n        # Edge features are temporal averages of P_hat entries\n        P_hat_t = P_hat_history[t]\n        edge_index = (P_hat_t.abs() > epsilon_S).nonzero().t()\n        edge_features = []\n        for src, dest in edge_index.t():\n            p_features = []\n            for window in lookback_windows:\n                avg_p = np.mean([P_hat_history[t-k][src, dest] for k in range(window)])\n                p_features.append(avg_p)\n            edge_features.append(p_features)\n        edge_features = torch.tensor(edge_features, dtype=torch.float)\n\n        fxsa_graph_sequence.append(Data(x=node_features, edge_index=edge_index, edge_attr=edge_features))\n\n    return fxsa_graph_sequence\n\ndummy_predictions = df_full[get_all_fx_tickers(currencies + ['USD'])]\nfxsa_graph_sequence = generate_fxsa_graph_sequence(dummy_predictions, currencies + ['USD'], lookback_windows)\nprint(f"Generated {len(fxsa_graph_sequence)} FXSA graphs for the time series.")\nif fxsa_graph_sequence:\n    print("Example FXSA graph data object:")\n    print(fxsa_graph_sequence[0])

## 6. FXSA GNN Model Implementation\n\nThis section defines the GNN for the FXSA task (`g_S`). As per the paper, it uses the same architecture as the FXRP GNN, but operates on the influence graph.

In [ ]:
# The FXSA_GNN has the same architecture as the FXRP_GNN.\nFXSA_GNN = FXRP_GNN \n\n# To instantiate it, we would need the feature dimensions from the FXSA graph sequence.\nif fxsa_graph_sequence:\n    fxsa_model = FXSA_GNN(num_node_features=fxsa_graph_sequence[0].num_node_features, \n                          num_edge_features=fxsa_graph_sequence[0].num_edge_features, \n                          hidden_dim=64, \n                          num_layers=3)\n    print("FXSA GNN Model Architecture:")\n    print(fxsa_model)

## 7. Constraint Handling Function (h_SO)\n\nThis section implements the post-processing function `h_SO`, which ensures that the output of the FXSA GNN satisfies the trading constraints.

In [ ]:
def post_process_trading_quantities(u_prime, P_hat):\n    """\n    This function takes the raw output of the FXSA GNN and applies the projection\n    and normalization to get the final trading quantities.\n    """\n    # Project onto the valid flow space\n    u = torch.matmul(P_hat, u_prime.unsqueeze(-1)).squeeze(-1)\n    \n    # Apply ReLU and normalize\n    u_pos = F.relu(u)\n    sum_u_pos = torch.sum(u_pos, dim=-1, keepdim=True)\n    w = u_pos / (sum_u_pos + 1e-8) # Add epsilon to avoid division by zero\n    \n    return w

## 8. Next Steps\n\nThis notebook has successfully fetched, preprocessed, and structured the data for both the FXRP and FXSA models. The GNN architectures and the constraint handling function are also implemented. The final steps are:\n\n1.  **Run FXRP Model:** Get predictions `X_hat` from the `FXRP_GNN`.\n2.  **Generate FXSA Graph:** Use the predictions to generate the FXSA graph sequence.\n3.  **Implement Training Loops:** Train both the `FXRP_GNN` and the `FXSA_GNN`.